# E2.8 · Auditability of autonomous action

**Function E — AI Governance for Agentic Systems → Building the Governance Platform — Regulatory and Compliance**  ·  *Security of AI*

Builds on **[E2.7 · Documentation that survives supervision](https://spbreed.github.io/cyber-commons/lessons/E2.7.html)**.

| | |
|---|---|
| Tools used | Keycloak |

## What this lesson is

**What it covers.** Produce an audit trail from the A2 chain that names authority at every hop.

**Why a security engineer needs it.** No trail showing under whose authority the agent acted. The control it builds is: the delegation chain *is* the audit trail.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

"Why did it do that?" is a question with a legal deadline attached. Logging designed forwards records what was convenient; logging designed backwards from the auditor's question records what is needed.

> **At CyberTravels.** “Why did CyberTravels issue that refund?” has a legal deadline attached. Logging designed backwards from that question records the booking note; logging designed forwards records the HTTP call. R11.

## 2 · The framework

```
   design the log backwards from the question

   auditor asks            log must contain
   why this decision?  ->  the input that motivated it
   on whose behalf?    ->  principal + delegation chain
   under what rule?    ->  policy version at that moment
   who reviewed it?    ->  approver identity and what they saw

   forwards-designed logs record what was convenient
```

Auditability of autonomous action reduces to one question:

> For any single action, can you produce **who caused it** and **what they were
> allowed to do**?

Answering it needs two capabilities that must both be present at the moment the
action happens, because neither can be reconstructed afterwards:

- **Attribution** — the acting identity, the principal, and the chain between
  them (A2.5, EV-1).
- **Replay** — the prompts, the tool results, the pinned model version and the
  seed (D2.5).

Attribution without replay tells you who acted but not why. Replay without
attribution tells you what happened but not on whose authority. Regulators and
auditors ask both, usually in that order.

## 3 · The procedure, as a skill

A record can name the acting identity, the principal, the chain and the scopes, be internally consistent, and be false. The skill scores three cases and adds the check that separates delegation from impersonation.

In [ ]:
# skills/regulatory/autonomous-action-auditability/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: autonomous-action-auditability
description: >-
  Decide whether an audit record makes an autonomous action answerable, and
  demonstrate a record that is complete, internally consistent and false. Use
  when an audit trail is being accepted as evidence of what an agent did.
allowed-tools: Read, Grep, Glob
---

# Complete, consistent, and false

Auditability of autonomous action is not "is there a record". A record can name
the acting identity, the principal, the delegation chain and the scopes, be
internally consistent, and still be false — because the token it rests on was
impersonated rather than delegated. Answerable requires the record *and* a way
to tell those apart.

## When to use this

When designing audit records for agent actions, and when one is offered as
evidence in an investigation or to an assessor.

## Procedure

**1 — Define answerable as a set of fields.** Acting identity, principal,
delegation chain, scopes exercised, and whether the run is replayable. Fewer
than all of them and some question is unanswerable; say which.

**2 — Score a complete record against the definition.** This is the good case
and it should pass — establishing that the definition is achievable rather than
aspirational.

**3 — Construct the impersonation case.** A token where the actor equals the
subject rather than being nested under it. The record is complete and consistent
and the attribution is wrong. Show that it scores identically on completeness.

**4 — Add the check that separates them.** Delegation has a distinct actor claim
nested under the subject; impersonation does not. The record must carry that
distinction, or completeness is all you can ever measure.

**5 — Test the no-replay case.** A record whose run cannot be replayed answers
what happened and never why. Mark it partially answerable rather than
answerable.

## Output contract

```json
{
  "definition": {"fields": ["str"], "replayable_required": true},
  "cases": [{"name": "complete|impersonated|no_replay", "fields_present": ["str"],
             "consistent": true, "answerable": false, "why": "str"}],
  "distinguisher": {"check": "str", "present": false}
}
```

## Failure modes

- **Measuring completeness.** The false record is complete.
- **Accepting a matching subject as delegation.** Impersonation matches too.
- **Calling a non-replayable record answerable.** It answers what, not why.
"""

In [ ]:
# Execute the skill above, using the shared runtime rather than a copy.
import glob, os, shutil, sys

# Make the shared runtime importable, then import it. On Kaggle an attached
# kernel is mounted as __script__.py — not on sys.path and not named after the
# kernel — so copy it to the name it is imported by. Locally it is already a
# file of that name in the repository.
_k = glob.glob("/kaggle/input/**/cyber-commons-skill-runtime/__script__.py", recursive=True)
if _k:
    shutil.copy(_k[0], "cyber_commons_skill_runtime.py")
sys.path[:0] = [".", "skills/_runtime", "../skills/_runtime", "../../skills/_runtime"]

from cyber_commons_skill_runtime import run_skill

# Split skills/regulatory/autonomous-action-auditability/SKILL.md into the two halves an agent uses —
# the frontmatter it routes on, and the body it follows.
meta, body = run_skill(SKILL_MD)

In [ ]:
# skills/regulatory/autonomous-action-auditability/scripts/autonomous_action_auditability.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Decide whether an audit record makes an autonomous action answerable, and show a complete, consistent, false record.

This is the executable half of the `autonomous-action-auditability` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

import time
from dataclasses import dataclass, field

@dataclass
class Token:
    sub: str; actor: str; scopes: set; act: dict = None
    def chain(self):
        out, node = [], self.act
        while node: out.append(node["actor"]); node = node.get("act")
        c = list(reversed(out)) + [self.actor]
        if c[0] != self.sub: c.insert(0, self.sub)
        return c

@dataclass
class Replay:
    prompts: list = field(default_factory=list)
    tool_results: list = field(default_factory=list)
    model_version: str = ""
    seed: object = None
    def replayable(self):
        missing = [n for n, v in (("prompts", self.prompts),
                                  ("tool results", self.tool_results),
                                  ("model version", self.model_version),
                                  ("seed", self.seed is not None)) if not v]
        return (not missing), missing

def audit_record(action, token, replay):
    ok, missing = replay.replayable()
    return {"action": action,
            "acting_identity": token.actor,
            "on_behalf_of": token.sub,
            "chain": " → ".join(token.chain()),
            "scopes_held": sorted(token.scopes),
            "replayable": ok,
            "replay_gaps": missing,
            "answerable": token.actor != token.sub and ok}

GOOD_TOKEN = Token("dana@corp", "patch-agent", {"repo:read", "repo:write"},
                   {"actor": "orchestrator", "act": None})
GOOD_REPLAY = Replay(["fix SEC-4471"], ["file contents…"], "glm-4.6@2026-07-14", 42)

r = audit_record("merge_pr #8812", GOOD_TOKEN, GOOD_REPLAY)
for k, v in r.items(): print(f"{k:18s}{v}")

IMPERSONATED = Token("dana@corp", "dana@corp", {"repo:write"}, None)
NO_REPLAY    = Replay(["fix SEC-4471"], ["file contents…"], "", None)

CASES = {
 "complete":                     (GOOD_TOKEN,  GOOD_REPLAY),
 "attribution broken":           (IMPERSONATED, GOOD_REPLAY),
 "replay incomplete":            (GOOD_TOKEN,  NO_REPLAY),
 "neither":                      (IMPERSONATED, NO_REPLAY),
}
print(f"{'case':22s}{'who acted':14s}{'replayable':12s}{'answerable':>12}")
print("-" * 62)
for name, (tok, rep) in CASES.items():
    r = audit_record("merge_pr #8812", tok, rep)
    print(f"{name:22s}{r['acting_identity']:14s}{str(r['replayable']):12s}"
          f"{str(r['answerable']):>12}")

r = audit_record("merge_pr #8812", IMPERSONATED, GOOD_REPLAY)
print(f"\nattribution-broken record: acting_identity={r['acting_identity']}")
print("The record is complete, internally consistent, and false. It says a human")
print("merged a pull request she never saw.")
assert not r["answerable"]

def auditability_drill(records, sample_size=3):
    """Pick actions at random and try to produce the full record for each."""
    results = []
    for i, (action, tok, rep) in enumerate(records[:sample_size], 1):
        r = audit_record(action, tok, rep)
        gaps = []
        if r["acting_identity"] == r["on_behalf_of"]:
            gaps.append("acting identity not distinguishable from the principal")
        gaps += [f"replay missing {m}" for m in r["replay_gaps"]]
        results.append({"n": i, "action": action, "complete": not gaps, "gaps": gaps})
    passed = sum(r["complete"] for r in results)
    return results, passed, len(results)

SAMPLE = [
 ("merge_pr #8812", GOOD_TOKEN, GOOD_REPLAY),
 ("deploy prod",    IMPERSONATED, GOOD_REPLAY),
 ("rotate secret",  GOOD_TOKEN, NO_REPLAY),
]
rows, passed, total = auditability_drill(SAMPLE)
for r in rows:
    print(f"{r['n']}. {r['action']:18s}{'COMPLETE' if r['complete'] else 'INCOMPLETE'}")
    for g in r["gaps"]: print(f"      ⚠ {g}")
print(f"\nauditability: {passed}/{total} sampled actions fully answerable "
      f"({passed/total:.0%})")
print("\nRun this as a drill, quarterly, on randomly chosen production actions.")
print("The percentage is the number that goes in the evidence pack — and it is")
print("far more persuasive than a statement that logging is comprehensive.")
assert passed < total

## What you just proved

The complete record names the acting identity, principal, chain and scopes and is replayable, so it is answerable. Impersonation produces a complete, consistent and false record attributing the merge to the human. Missing replay fields break the other half. The drill reports 1 of 3 sampled actions fully answerable.

## Your turn

Run the drill on three real production actions from last week. The field you cannot fill is your auditability gap, stated precisely — and a number like "1 of 3" is far more useful to a supervisor than a paragraph about comprehensive logging.

---

**Next → [E2.9 · Regulator and auditor conversations](https://spbreed.github.io/cyber-commons/lessons/E2.9.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/E2.8.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/E2.8.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*